# Alignement automatique des vers

Ce notebook prend en entrée des témoins TEI-XML encodés par vers et reconstruit une numérotation commune des vers à l’échelle de l’ensemble de la tradition.

Le workflow est le suivant :

1. charger tous les témoins depuis un dossier (ici cliges8_sample) ;
2. extraire les vers dans l’ordre du document ;
3. aligner automatiquement les témoins sur la base de la similarité entre vers ;
4. construire une numérotation commune unique ;
5. insérer des éléments de vers vides lorsqu’un témoin n’atteste pas de vers pour une position partagée ;
6. exporter :
   - une table d’alignement (`.tsv`) ;
   - une description structurée de l’alignement (`.json`) ;
   - des fichiers TEI-XML renumérotés pour chaque témoin.

> Toute position de vers attestée quelque part dans la tradition reçoit un numéro commun. Si un témoin omet un vers présent dans d’autres, l’export TEI correspondant contient un élément `<l>` vide portant ce numéro commun.

## Configurations

In [1]:
from pathlib import Path
from datetime import datetime
import json
import re
import unicodedata
from collections import Counter
from copy import deepcopy

import pandas as pd
from lxml import etree

In [2]:
repo_root = Path.cwd()

input_dir = repo_root / "cliges8_sample"
output_dir = repo_root / "alignment_exports"

pivot_siglum = "A"

selected_witnesses = None
min_match_score = 0.38
gap_penalty = -0.45

output_dir.mkdir(parents=True, exist_ok=True)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

## Dossiers

In [3]:
for p in sorted(repo_root.iterdir()):
    if p.is_dir():
        xml_files = sorted(p.glob("*.xml"))
        if xml_files:
            print(f"{p.name}/  ({len(xml_files)} xml files)")

cliges/  (3 xml files)
cliges2/  (10 xml files)
cliges3/  (7 xml files)
cliges4/  (8 xml files)
cliges5/  (9 xml files)
cliges6/  (7 xml files)
cliges6bis/  (6 xml files)
cliges7/  (6 xml files)
cliges8/  (6 xml files)
cliges8_sample/  (6 xml files)
cliges_doc_travail/  (4 xml files)


## Fonctions utilitaires pour la normalisation des vers

In [4]:
TEI_NS = "http://www.tei-c.org/ns/1.0"
XML_NS = "http://www.w3.org/XML/1998/namespace"
NS = {"tei": TEI_NS}


def normalize_graphies(text):
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.lower()
    text = text.replace("j", "i").replace("v", "u")
    text = text.replace("y", "i")
    text = text.replace("ç", "c")
    text = text.replace("œ", "oe").replace("æ", "ae")
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\bqu\b", "cu", text)
    text = re.sub(r"qu", "c", text)
    text = re.sub(r"ph", "f", text)
    text = re.sub(r"th", "t", text)
    text = re.sub(r"ch", "c", text)
    text = re.sub(r"gn", "n", text)
    text = re.sub(r"lh", "l", text)
    text = re.sub(r"dh", "d", text)
    text = re.sub(r"([bcdfghklmnpqrstxz])\1+", r"\1", text)
    text = re.sub(r"u+", "u", text)
    text = re.sub(r"i+", "i", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def consonant_skeleton(text):
    normalized = normalize_graphies(text)
    skeleton = re.sub(r"[aeiou\s]", "", normalized)
    skeleton = re.sub(r"(.)\1+", r"\1", skeleton)
    return skeleton


def normalized_tokens(text):
    normalized = normalize_graphies(text)
    return [tok for tok in normalized.split() if tok]


def tail_signature(text, n=10):
    normalized = normalize_graphies(text).replace(" ", "")
    return normalized[-n:]


def jaccard(items_a, items_b):
    set_a = set(items_a)
    set_b = set(items_b)
    if not set_a and not set_b:
        return 1.0
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)


def multiset_overlap(tokens_a, tokens_b):
    if not tokens_a and not tokens_b:
        return 1.0
    if not tokens_a or not tokens_b:
        return 0.0
    c_a = Counter(tokens_a)
    c_b = Counter(tokens_b)
    common = sum((c_a & c_b).values())
    total = max(sum(c_a.values()), sum(c_b.values()))
    return common / total if total else 0.0


def char_similarity(a, b):
    if not a and not b:
        return 1.0
    return SequenceMatcher(None, a, b).ratio()


from difflib import SequenceMatcher


def verse_similarity(text_a, text_b):
    norm_a = normalize_graphies(text_a)
    norm_b = normalize_graphies(text_b)

    tok_a = normalized_tokens(text_a)
    tok_b = normalized_tokens(text_b)

    skeleton_a = consonant_skeleton(text_a)
    skeleton_b = consonant_skeleton(text_b)

    tail_a = tail_signature(text_a)
    tail_b = tail_signature(text_b)

    scores = {
        "char": char_similarity(norm_a, norm_b),
        "tokens": jaccard(tok_a, tok_b),
        "multiset": multiset_overlap(tok_a, tok_b),
        "skeleton": char_similarity(skeleton_a, skeleton_b),
        "tail": char_similarity(tail_a, tail_b),
        "length": 1.0 - min(abs(len(tok_a) - len(tok_b)) / max(len(tok_a), len(tok_b), 1), 1.0),
    }

    score = (
        0.28 * scores["char"] +
        0.18 * scores["tokens"] +
        0.18 * scores["multiset"] +
        0.20 * scores["skeleton"] +
        0.10 * scores["tail"] +
        0.06 * scores["length"]
    )
    return score, scores

## Représentation des témoins

In [5]:
class VerseRecord:
    def __init__(self, element, index, witness_siglum):
        self.element = element
        self.index = index
        self.witness_siglum = witness_siglum
        self.original_n = element.get("n") or element.get(f"{{{XML_NS}}}id") or ""
        self.text = " ".join(" ".join(element.itertext()).split())

    def clone(self):
        return deepcopy(self.element)


class Witness:
    def __init__(self, path):
        self.path = Path(path)
        self.tree = etree.parse(str(self.path))
        self.root = self.tree.getroot()
        self.siglum = self._resolve_siglum()
        self.lines = self._extract_lines()

    def _resolve_siglum(self):
        siglum = self.root.xpath("string(//*[local-name()='div'][1]/@id)")
        siglum = siglum.strip()
        return siglum if siglum else self.path.stem

    def _extract_lines(self):
        lines = self.root.xpath("//*[local-name()='l']")
        return [VerseRecord(line, idx + 1, self.siglum) for idx, line in enumerate(lines)]

    def verse_texts(self):
        return [line.text for line in self.lines]

    def __len__(self):
        return len(self.lines)


def load_witnesses(input_dir, selected_witnesses=None):
    input_dir = Path(input_dir)
    paths = sorted(
        path for path in input_dir.glob("*.xml")
        if not path.name.startswith("._")
    )
    witnesses = [Witness(path) for path in paths]
    if selected_witnesses:
        selected = set(selected_witnesses)
        witnesses = [w for w in witnesses if w.siglum in selected]
    return witnesses

## Chargement de corpus

In [6]:
witnesses = load_witnesses(input_dir, selected_witnesses=selected_witnesses)
sigla = [w.siglum for w in witnesses]

if not witnesses:
    raise ValueError(f"No XML witnesses found in {input_dir}")

print("Witnesses loaded:")
for w in witnesses:
    print(f"- {w.siglum}: {len(w)} verses")

if pivot_siglum not in sigla:
    pivot_siglum = witnesses[0].siglum

print(f"\nPivot witness: {pivot_siglum}")

Witnesses loaded:
- A: 96 verses
- B: 98 verses
- C: 98 verses
- P: 96 verses
- R: 98 verses
- S: 98 verses

Pivot witness: A


## Alignement de la séquence sur la structure commune actuelle

In [7]:
class AlignmentColumn:
    def __init__(self):
        self.entries = {}

    def set(self, siglum, verse_record):
        self.entries[siglum] = verse_record

    def get(self, siglum):
        return self.entries.get(siglum)

    def occupied_texts(self):
        return [record.text for record in self.entries.values() if record is not None and record.text.strip()]

    def representative_text(self):
        texts = self.occupied_texts()
        if not texts:
            return ""
        if len(texts) == 1:
            return texts[0]
        return max(texts, key=lambda t: sum(verse_similarity(t, other)[0] for other in texts))

    def score_against(self, verse_record):
        texts = self.occupied_texts()
        if not texts:
            return 0.0, {}
        scored = [verse_similarity(verse_record.text, text) for text in texts]
        scored.sort(key=lambda item: item[0], reverse=True)
        return scored[0]

    def is_empty(self):
        return not any(v is not None for v in self.entries.values())

In [8]:
def align_witness_to_columns(columns, witness, gap_penalty=-0.45, min_match_score=0.38):
    m = len(columns)
    n = len(witness.lines)

    dp = [[0.0] * (n + 1) for _ in range(m + 1)]
    bt = [[None] * (n + 1) for _ in range(m + 1)]

    for i in range(1, m + 1):
        dp[i][0] = dp[i - 1][0] + gap_penalty
        bt[i][0] = "up"

    for j in range(1, n + 1):
        dp[0][j] = dp[0][j - 1] + gap_penalty
        bt[0][j] = "left"

    score_cache = {}
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            column = columns[i - 1]
            verse = witness.lines[j - 1]
            if (i - 1, j - 1) not in score_cache:
                score_cache[(i - 1, j - 1)] = column.score_against(verse)
            score, detail = score_cache[(i - 1, j - 1)]

            match_score = score if score >= min_match_score else (gap_penalty - 0.15)

            diag = dp[i - 1][j - 1] + match_score
            up = dp[i - 1][j] + gap_penalty
            left = dp[i][j - 1] + gap_penalty

            best = max(diag, up, left)
            dp[i][j] = best
            bt[i][j] = "diag" if best == diag else ("up" if best == up else "left")

    i, j = m, n
    aligned_columns = []

    while i > 0 or j > 0:
        move = bt[i][j]
        if move == "diag":
            base_column = deepcopy(columns[i - 1])
            score, detail = score_cache[(i - 1, j - 1)]
            verse = witness.lines[j - 1]
            if score >= min_match_score:
                base_column.set(witness.siglum, verse)
                aligned_columns.append(base_column)
                i -= 1
                j -= 1
            else:
                aligned_columns.append(base_column)
                new_column = AlignmentColumn()
                new_column.set(witness.siglum, verse)
                aligned_columns.append(new_column)
                i -= 1
                j -= 1
        elif move == "up":
            aligned_columns.append(deepcopy(columns[i - 1]))
            i -= 1
        elif move == "left":
            new_column = AlignmentColumn()
            new_column.set(witness.siglum, witness.lines[j - 1])
            aligned_columns.append(new_column)
            j -= 1
        else:
            break

    aligned_columns.reverse()
    return aligned_columns


def build_master_alignment(witnesses, pivot_siglum="A", gap_penalty=-0.45, min_match_score=0.38):
    witness_map = {w.siglum: w for w in witnesses}
    pivot = witness_map[pivot_siglum]

    columns = []
    for verse in pivot.lines:
        col = AlignmentColumn()
        col.set(pivot.siglum, verse)
        columns.append(col)

    other_sigla = [w.siglum for w in witnesses if w.siglum != pivot_siglum]
    for siglum in other_sigla:
        columns = align_witness_to_columns(
            columns=columns,
            witness=witness_map[siglum],
            gap_penalty=gap_penalty,
            min_match_score=min_match_score
        )

    return columns

## Construction de la numérotation commune

In [9]:
columns = build_master_alignment(
    witnesses=witnesses,
    pivot_siglum=pivot_siglum,
    gap_penalty=gap_penalty,
    min_match_score=min_match_score
)

print(f"Shared verse positions: {len(columns)}")

Shared verse positions: 100


In [10]:
def alignment_dataframe(columns, sigla):
    rows = []
    for idx, col in enumerate(columns, start=1):
        row = {"shared_n": idx}
        for siglum in sigla:
            record = col.get(siglum)
            row[f"{siglum}_index"] = record.index if record else None
            row[f"{siglum}_text"] = record.text if record else ""
            row[f"{siglum}_original_n"] = record.original_n if record else ""
        rows.append(row)
    return pd.DataFrame(rows)

alignment_df = alignment_dataframe(columns, sigla)
alignment_df.head()

,shared_n,A_index,A_text,A_original_n,B_index,B_text,B_original_n,C_index,C_text,C_original_n,P_index,P_text,P_original_n,R_index,R_text,R_original_n,S_index,S_text,S_original_n
0,1,1.0,Soredamors prant la chemise,,1.0,La roine prent la cemise,,1.0,La reine prist la chemise,,1.0,La roine prent la cemise,,1.0,La roine prist la chemise,,1.0,La reine prent la chemise,
1,2,2.0,Si l'a Alixandre tramise,,2.0,Si l'a alixandre tramise,,2.0,Si l'a Alixandre tramise,,2.0,Si l'a Alixandre tramise,,2.0,Si l'a alixandre tramise,,2.0,Si l'a Alixandre tramisse,
2,3,3.0,Et Dex com grant joie an eust,,3.0,Ha! Dex! Com grant joie en eust,,3.0,Et Dex com grant joie en eust,,3.0,Ha Diex com grant joie en eust,,3.0,Ha dex com grant joie en eust,,3.0,He Dex com grant joie en aust,
3,4,4.0,Alixandre s'il le seust,,4.0,[192a] alixandre s'il le seust,,4.0,Alixandre se il seust,,4.0,Alixandre se il seust,,4.0,Alixande s'il seust,,4.0,Alixandre se il lo saust,
4,5,5.0,Que la reine li anvoie,,5.0,Que la roine li envoie!,,5.0,Que la reine li envoie,,5.0,Que la roine li envoie,,5.0,Que la roine li envoie,,5.0,Que la roine li envoie,


## Export de l'alignement

In [11]:
alignment_tsv_path = output_dir / "alignment_table.tsv"
alignment_json_path = output_dir / "alignment_table.json"

alignment_df.to_csv(alignment_tsv_path, sep="\t", index=False)

alignment_json = []
for idx, col in enumerate(columns, start=1):
    item = {"shared_n": idx, "witnesses": {}}
    for siglum in sigla:
        record = col.get(siglum)
        item["witnesses"][siglum] = {
            "index": record.index if record else None,
            "original_n": record.original_n if record else "",
            "text": record.text if record else ""
        }
    alignment_json.append(item)

alignment_json_path.write_text(
    json.dumps(alignment_json, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

alignment_tsv_path, alignment_json_path

(PosixPath('/Users/benedettasalvati/Desktop/Biblissima_Cliges/alignment_exports/alignment_table.tsv'),
 PosixPath('/Users/benedettasalvati/Desktop/Biblissima_Cliges/alignment_exports/alignment_table.json'))

## Réécriture des fichiers TEI d'origine

In [12]:
def make_empty_line():
    return etree.Element(f"{{{TEI_NS}}}l")


def renumbered_tei_for_witness(columns, witness, shared_attr="n"):
    root = etree.Element(f"{{{TEI_NS}}}TEI", nsmap={None: TEI_NS, "xml": XML_NS})

    teiHeader = etree.SubElement(root, f"{{{TEI_NS}}}teiHeader")
    fileDesc = etree.SubElement(teiHeader, f"{{{TEI_NS}}}fileDesc")
    titleStmt = etree.SubElement(fileDesc, f"{{{TEI_NS}}}titleStmt")
    title = etree.SubElement(titleStmt, f"{{{TEI_NS}}}title")
    title.text = f"Renumbered witness {witness.siglum}"

    publicationStmt = etree.SubElement(fileDesc, f"{{{TEI_NS}}}publicationStmt")
    p_pub = etree.SubElement(publicationStmt, f"{{{TEI_NS}}}p")
    p_pub.text = "Automatically generated shared numbering."

    sourceDesc = etree.SubElement(fileDesc, f"{{{TEI_NS}}}sourceDesc")
    p_source = etree.SubElement(sourceDesc, f"{{{TEI_NS}}}p")
    p_source.text = f"Derived from {witness.path.name}."

    text = etree.SubElement(root, f"{{{TEI_NS}}}text")
    body = etree.SubElement(text, f"{{{TEI_NS}}}body")
    lg = etree.SubElement(body, f"{{{TEI_NS}}}lg")
    lg.set("type", "aligned")

    for shared_n, col in enumerate(columns, start=1):
        record = col.get(witness.siglum)
        if record is None:
            line = make_empty_line()
        else:
            line = record.clone()

        line.set(shared_attr, str(shared_n))
        line.attrib.pop(f"{{{XML_NS}}}id", None)
        lg.append(line)

    return root

In [13]:
renumbered_dir = output_dir / "renumbered_witnesses"
renumbered_dir.mkdir(parents=True, exist_ok=True)

renumbered_paths = []

witness_map = {w.siglum: w for w in witnesses}

for siglum in sigla:
    new_root = renumbered_tei_for_witness(columns, witness_map[siglum], shared_attr="n")
    out_path = renumbered_dir / f"{siglum}_renumbered.xml"
    etree.ElementTree(new_root).write(
        str(out_path),
        encoding="utf-8",
        xml_declaration=True,
        pretty_print=True
    )
    renumbered_paths.append(out_path)

renumbered_paths[:5]

[PosixPath('/Users/benedettasalvati/Desktop/Biblissima_Cliges/alignment_exports/renumbered_witnesses/A__renumbered.xml'),
 PosixPath('/Users/benedettasalvati/Desktop/Biblissima_Cliges/alignment_exports/renumbered_witnesses/B__renumbered.xml'),
 PosixPath('/Users/benedettasalvati/Desktop/Biblissima_Cliges/alignment_exports/renumbered_witnesses/C__renumbered.xml'),
 PosixPath('/Users/benedettasalvati/Desktop/Biblissima_Cliges/alignment_exports/renumbered_witnesses/P__renumbered.xml'),
 PosixPath('/Users/benedettasalvati/Desktop/Biblissima_Cliges/alignment_exports/renumbered_witnesses/R__renumbered.xml')]

## Aperçu des interventions

In [14]:
for siglum in sigla:
    missing = alignment_df[f"{siglum}_index"].isna().sum()
    print(f"{siglum}: {missing} empty verses inserted")

A: 4 empty verses inserted
B: 2 empty verses inserted
C: 2 empty verses inserted
P: 4 empty verses inserted
R: 2 empty verses inserted
S: 2 empty verses inserted


## Collation sur la base de l'alignement automatique

In [15]:
from collatex import collate


def line_to_collatex_tokens(line_element):
    text = " ".join(" ".join(line_element.itertext()).split())
    if not text:
        return []
    tokens = []
    for token in text.split():
        tokens.append({"t": token, "n": normalize_graphies(token)})
    return tokens


def collate_shared_position(columns, shared_n):
    col = columns[shared_n - 1]
    witnesses_json = []

    for siglum in sigla:
        record = col.get(siglum)
        if record is None:
            continue
        tokens = line_to_collatex_tokens(record.element)
        if tokens:
            witnesses_json.append({"id": siglum, "tokens": tokens})

    if not witnesses_json:
        return None

    return collate({"witnesses": witnesses_json}, output="xml", segmentation=False)

In [16]:
def build_collation_tei(columns):
    root = etree.Element(f"{{{TEI_NS}}}TEI", nsmap={None: TEI_NS, "xml": XML_NS})

    teiHeader = etree.SubElement(root, f"{{{TEI_NS}}}teiHeader")
    fileDesc = etree.SubElement(teiHeader, f"{{{TEI_NS}}}fileDesc")
    titleStmt = etree.SubElement(fileDesc, f"{{{TEI_NS}}}titleStmt")
    title = etree.SubElement(titleStmt, f"{{{TEI_NS}}}title")
    title.text = "Collation on reconstructed shared numbering"

    publicationStmt = etree.SubElement(fileDesc, f"{{{TEI_NS}}}publicationStmt")
    p_pub = etree.SubElement(publicationStmt, f"{{{TEI_NS}}}p")
    p_pub.text = "Automatically generated."

    sourceDesc = etree.SubElement(fileDesc, f"{{{TEI_NS}}}sourceDesc")
    p_source = etree.SubElement(sourceDesc, f"{{{TEI_NS}}}p")
    p_source.text = "Collation computed after automatic verse alignment and shared numbering."

    text = etree.SubElement(root, f"{{{TEI_NS}}}text")
    body = etree.SubElement(text, f"{{{TEI_NS}}}body")
    div = etree.SubElement(body, f"{{{TEI_NS}}}div")
    div.set("type", "collation")

    for shared_n in range(1, len(columns) + 1):
        xml_output = collate_shared_position(columns, shared_n)
        line = etree.SubElement(div, f"{{{TEI_NS}}}l")
        line.set("n", str(shared_n))

        if xml_output:
            wrapped = etree.fromstring(f"<wrapper>{xml_output}</wrapper>")
            for child in wrapped:
                line.append(child)

    return root

In [17]:
collation_root = build_collation_tei(columns)
collation_path = output_dir / "collation_reconstructed_numbering.tei.xml"

etree.ElementTree(collation_root).write(
    str(collation_path),
    encoding="utf-8",
    xml_declaration=True,
    pretty_print=True
)

collation_path

PosixPath('/Users/benedettasalvati/Desktop/Biblissima_Cliges/alignment_exports/collation_reconstructed_numbering.tei.xml')

## Résumé

In [18]:
summary = {
    "input_dir": str(input_dir),
    "pivot_siglum": pivot_siglum,
    "witnesses": sigla,
    "shared_positions": len(columns),
    "alignment_tsv": str(alignment_tsv_path),
    "alignment_json": str(alignment_json_path),
    "renumbered_dir": str(renumbered_dir),
    "collation_tei": str(collation_path),
    "min_match_score": min_match_score,
    "gap_penalty": gap_penalty,
}

summary_path = output_dir / "summary.json"
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

summary_path

PosixPath('/Users/benedettasalvati/Desktop/Biblissima_Cliges/alignment_exports/summary.json')